In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
import os

In [0]:
if not any(m.mountPoint == "/mnt/fixturedata" for m in dbutils.fs.mounts()):
    storage_account = "laligadatadl" 
    container = "fixturedata"
    access_key = "Add your key"

    configuration = { f"fs.azure.account.key.{storage_account}.blob.core.windows.net": access_key }
    

    dbutils.fs.mount(
        source = f"wasbs://{container}@{storage_account}.blob.core.windows.net",
        mount_point = "/mnt/fixturedata",
        extra_configs = configuration
    )

   


In [0]:
dbutils.widgets.text("fixtureID", "")
fixture_id = dbutils.widgets.get("fixtureID")
#fixture_id = "720774"

dbutils.widgets.text("seasonYear","")
season_year = dbutils.widgets.get("seasonYear")
#season_year = "2021"

dbutils.widgets.text("sinkFileNumber","")
sink_file_number = dbutils.widgets.get("sinkFileNumber")
#sink_file_number = 1

In [0]:
df = spark.read.option("multiline", "true").json(f"/mnt/fixturedata/{season_year}/fixture_{fixture_id}.json")
#display(df)

In [0]:
df_league = df.select(explode(df.response).alias("league"))
#display(df_league)
df_league_new = df_league.select(col("league.league.id").alias("league_id"),
                             col("league.league.name").alias("league_name"),
                             col("league.league.country").alias("league_country"),
                             col("league.league.season").alias("season_year"))

#display(df_league_new)

In [0]:
df_fixture = df.select(explode(df.response).alias("fixture"))
#display(df_fixture)

In [0]:
df_statistics = df_fixture.select(col("fixture.league.id").alias("league_id"),
                                  col("fixture.fixture.id").alias("fixture_id"),
                                  col("fixture.league.name").alias("league_name"),
                                  col("fixture.league.country").alias("league_country"),
                                  col("fixture.league.season").alias("season_id"),
                                  col("fixture.statistics.team.id")[0].alias("team_id_home"),
                                  col("fixture.statistics.team.name")[0].alias("team_name_home"),
                                  col("fixture.statistics.statistics")[0][0].value.alias("shots_on_target_home"),
                                  col("fixture.statistics.statistics")[0][1].value.alias("shots_off_target_home"),
                                  col("fixture.statistics.statistics")[0][2].value.alias("total_shots_home"),
                                  col("fixture.statistics.statistics")[0][3].value.alias("shots_blocked_home"),
                                  col("fixture.statistics.statistics")[0][4].value.alias("shots_insidebox_home"),
                                  col("fixture.statistics.statistics")[0][5].value.alias("shots_outsidebox_home"),
                                  col("fixture.statistics.statistics")[0][6].value.alias("fouls_home"),
                                  col("fixture.statistics.statistics")[0][7].value.alias("corners_home"),
                                  col("fixture.statistics.statistics")[0][8].value.alias("offsides_home"),
                                  col("fixture.statistics.statistics")[0][9].value.alias("ball_possession_home"),
                                  col("fixture.statistics.statistics")[0][10].value.alias("yellow_cards_home"),
                                  col("fixture.statistics.statistics")[0][11].value.alias("red_cards_home"),
                                  col("fixture.statistics.statistics")[0][12].value.alias("saves_home"),
                                  col("fixture.statistics.statistics")[0][13].value.alias("passes_home"),
                                  col("fixture.statistics.statistics")[0][14].value.alias("accurate_passes_home"),
                                  col("fixture.statistics.statistics")[0][15].value.alias("pass_accuracy_home"),
                                  col("fixture.statistics.team.id")[1].alias("team_id_away"), 
                                  col("fixture.statistics.team.name")[1].alias("team_name_away"), 
                                  col("fixture.statistics.statistics")[1][0].value.alias("shots_on_target_away"),
                                  col("fixture.statistics.statistics")[1][1].value.alias("shots_off_target_away"),
                                  col("fixture.statistics.statistics")[1][2].value.alias("total_shots_away"),
                                  col("fixture.statistics.statistics")[1][3].value.alias("shots_blocked_away"),
                                  col("fixture.statistics.statistics")[1][4].value.alias("shots_insidebox_away"),
                                  col("fixture.statistics.statistics")[1][5].value.alias("shots_outsidebox_away"),    
                                  col("fixture.statistics.statistics")[1][6].value.alias("fouls_away"),
                                  col("fixture.statistics.statistics")[1][7].value.alias("corners_away"),
                                  col("fixture.statistics.statistics")[1][8].value.alias("offsides_away"),
                                  col("fixture.statistics.statistics")[1][9].value.alias("ball_possession_away"),
                                  col("fixture.statistics.statistics")[1][10].value.alias("yellow_cards_away"),
                                  col("fixture.statistics.statistics")[1][11].value.alias("red_cards_away"),
                                  col("fixture.statistics.statistics")[1][12].value.alias("saves_away"),
                                  col("fixture.statistics.statistics")[1][13].value.alias("passes_away"),
                                  col("fixture.statistics.statistics")[1][14].value.alias("accurate_passes_away"),
                                  col("fixture.statistics.statistics")[1][15].value.alias("pass_accuracy_away"),
                                  concat(col("fixture.league.id"),lit("-"),col("fixture.league.season"),lit("-"),col("fixture.fixture.id"),lit("-"),col("fixture.statistics.team.id")[0],lit("-"),col("fixture.statistics.team.id")[1]).alias("uq_id")
                                  )

df_statistics = df_statistics.na.fill({"yellow_cards_home": 0, "red_cards_home": 0, "yellow_cards_away": 0, "red_cards_away": 0, "shots_on_target_home": 0, "shots_off_target_home": 0, "total_shots_home": 0, "shots_blocked_home": 0, "shots_insidebox_home": 0, "shots_outsidebox_home": 0, "fouls_home": 0, "corners_home": 0, "offsides_home": 0, "ball_possession_home": 0, "saves_home": 0, "passes_home": 0, "accurate_passes_home": 0, "pass_accuracy_home": 0, "shots_on_target_away": 0, "shots_off_target_away": 0, "total_shots_away": 0, "shots_blocked_away": 0, "shots_insidebox_away": 0, "shots_outsidebox_away": 0, "fouls_away": 0, "corners_away": 0, "offsides_away": 0, "ball_possession_away": 0, "saves_away": 0, "passes_away": 0, "accurate_passes_away": 0, "pass_accuracy_away": 0})

#display(df_statistics)


if not spark.catalog._jcatalog.tableExists("df_statistics"):
    df_statistics.limit(0).write.mode("overwrite").saveAsTable("df_statistics")

df_statistics.write.mode("append").saveAsTable("df_statistics")


In [0]:
#_sqldf.write.option("header", "true").mode("overwrite").csv(f"/mnt/fixturedata/{season_year}/{fixture_id}_fixture.csv")
df_statistics.coalesce(1).write.option("header", "true").mode("append").csv(f"/mnt/fixturedata/{season_year}/list_{sink_file_number}_fixtures.csv")